In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from routellm.controller import Controller

In [5]:
# Load MMLU-Pro (query + options)
df = pd.read_parquet("../datasets/processed/mmlu_pro.parquet", columns=["query", "options"])
df = df.dropna(subset=["query"]).copy()


In [7]:
df.head( )

,query,options
0,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa..."
1,Managers are entrusted to run the company in t...,"[Shareholders, Diligence, Self-interest, Share..."
2,There are two main issues associated with ____...,"[Down, Autonomy, Remuneration, Benefit, Down, ..."
3,_______ locate morality beyond the sphere of r...,"[Ethical egoism, Ethics of duty, Postmodern et..."
4,Some of key differences between Islamic finan...,"[Interest, Certain, Assured, Both tangible and..."


In [12]:
def format_options(opts):
    # Handle list-like options first (including numpy arrays)
    if isinstance(opts, np.ndarray):
        opts = opts.tolist()
    if isinstance(opts, (list, tuple)):
        return "\n".join(f"{chr(65+i)}. {str(o)}" for i, o in enumerate(opts))

    # Scalars / missing values
    if opts is None:
        return ""
    if isinstance(opts, str):
        return opts.strip()
    if pd.isna(opts):
        return ""
    return str(opts)

In [ ]:
# Build routing prompt with BOTH query and options
df["options_text"] = df["options"].apply(format_options)
df["router_prompt"] = df.apply(
    lambda r: f"Question:\n{str(r['query']).strip()}\n\nOptions:\n{r['options_text']}".strip(),
    axis=1,
)

In [14]:
client = Controller(
    routers=["mf"],
    strong_model="gpt-4o",
    weak_model="gpt-4o-mini",
)

# Loa

In [15]:
df.head(10)

,query,options,options_text,router_prompt
0,"Typical advertising regulatory bodies suggest,...","[Safe practices, Fear, Jealousy, Trivial, Unsa...","A. Safe practices, Fear, Jealousy, Trivial\nB....",Question:\nTypical advertising regulatory bodi...
1,Managers are entrusted to run the company in t...,"[Shareholders, Diligence, Self-interest, Share...","A. Shareholders, Diligence, Self-interest\nB. ...",Question:\nManagers are entrusted to run the c...
2,There are two main issues associated with ____...,"[Down, Autonomy, Remuneration, Benefit, Down, ...","A. Down, Autonomy, Remuneration, Benefit\nB. D...",Question:\nThere are two main issues associate...
3,_______ locate morality beyond the sphere of r...,"[Ethical egoism, Ethics of duty, Postmodern et...",A. Ethical egoism\nB. Ethics of duty\nC. Postm...,Question:\n_______ locate morality beyond the ...
4,Some of key differences between Islamic finan...,"[Interest, Certain, Assured, Both tangible and...","A. Interest, Certain, Assured, Both tangible a...",Question:\nSome of key differences between Isl...
5,Which of the following are the three broad gr...,"[Organizational size, industry type, and geogr...","A. Organizational size, industry type, and geo...",Question:\nWhich of the following are the thre...
6,Pine and Gilmore (1999) derive four distinct ...,[Customer participation and environmental acqu...,A. Customer participation and environmental ac...,Question:\nPine and Gilmore (1999) derive four...
7,Which type of research methods are designed t...,"[Non-probability., Cross-sectional., Qualitati...",A. Non-probability.\nB. Cross-sectional.\nC. Q...,Question:\nWhich type of research methods are ...
8,Where the price is set low relative to the com...,"[Captive product pricing., High-low pricing., ...",A. Captive product pricing.\nB. High-low prici...,Question:\nWhere the price is set low relative...
9,"Once a train pulls out of a station, or an aer...","[Immeasurability., Impalpability., Variability...",A. Immeasurability.\nB. Impalpability.\nC. Var...,Question:\nOnce a train pulls out of a station...


In [ ]:





# (Optional) quick test size
df = df.head(10)



# Threshold for target strong-model routing share
target_pct = 0.25
threshold = float(np.percentile(scores, 100 - target_pct * 100))

# Save parquet
out = df[["query", "options", "router_prompt"]].copy()
out["strong_win_rate"] = scores
out["threshold"] = threshold
out["route_to_strong"] = out["strong_win_rate"] >= threshold

out_dir = Path("../results1/unified_baseline/mmlu_pro/routellm")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "routellm_scores.parquet"
out.to_parquet(out_path, index=False)

print(f"Scored {len(out)} MMLU-Pro prompts")
print(f"Threshold (25% strong): {threshold:.5f}")
print(f"Saved: {out_path}")

In [1]:
from routellm.controller import Controller
import json

/Users/poorna/Desktop/research_work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = Controller(
    routers=["mf"],
    strong_model="gpt-4o",
    weak_model="gpt-4o-mini",
)

# Loa

In [3]:
import numpy as np
import pandas as pd

# Load only the query column
queries = (
    pd.read_parquet("../datasets/processed/gaia.parquet", columns=["query"])["query"]
    .dropna()
    .astype(str)
    .str.strip()
)
